# Inspect Driver — Demo

Turn a model served through [`inspect_ai`](https://inspect.aisi.org.uk) into a
TransformerLens `HookedTransformer`: `run_with_cache`, named hook points, and
interventions all work over the Inspect boundary.

`boot_inspect` boots the model behind an `inspect_ai` provider and wraps it in a
bridge with the standard TL hook contract.

**Scope (v1):**
- Captures residual / attention / MLP outputs under their TransformerBridge names
  (the same canonical names the vLLM driver uses):
  `blocks.{i}.hook_in` (resid_pre) / `blocks.{i}.ln2.hook_in` (resid_mid) /
  `blocks.{i}.hook_out` (resid_post), `blocks.{i}.attn.hook_out`,
  `blocks.{i}.mlp.hook_out`. Head-split hooks (`hook_q/k/v/z`, `hook_pattern`)
  aren't available — use `boot_transformers()`.
- Which of those a given model actually serves is decided by a per-model structural
  self-check: `resid_mid` is gated on parallel / norm-variant architectures, and
  `attn_out`/`mlp_out` when their submodule isn't locatable. gpt2 (here) serves all five.
- Returns full-sequence logits, so `return_type="loss"` works too.

Runs on CPU; no GPU required.

## Setup

Install TransformerLens with the `inspect` extra (pulls in `inspect_ai`):

```bash
pip install "transformer_lens[inspect]"
```

In [1]:
import warnings
warnings.filterwarnings("ignore")
import torch

from transformer_lens.model_bridge.remote_bridge import RemoteBridge
from transformer_lens.model_bridge.transformer_bridge import TransformerBridge

MODEL = "gpt2"
PROMPT = "The quick brown fox"
torch.manual_seed(0)

## Step 1 — Boot the model through Inspect

`boot_inspect` resolves the architecture/config (no weights loaded TL-side), boots
the model behind an `inspect_ai` provider, and wraps it in a `RemoteBridge` with
the standard hook contract.

In [2]:
bridge = RemoteBridge.boot_inspect(MODEL)
print("bridge:", type(bridge).__name__)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

bridge: RemoteBridge


## Step 2 — Listing available hooks

`bridge.hook_dict` is the standard accessor for available hook points (the same one
every bridge exposes); `bridge.list_hooks()` shows only the hooks with a handler
currently attached. The canonical names are the **TransformerBridge-native** ones the
driver declares; which of those the provider can actually *fire* is driver-specific —
head-split hooks (`hook_q/k/v/z`, `hook_pattern`), `embed`, and `ln_final` aren't
served (they're in `non_fireable_hook_points`).

In [3]:
# `hook_dict` lists all available hook points (it also surfaces HookedTransformer-style
# aliases the bridge system registers for compatibility):
print("hook_dict entries:", len(bridge.hook_dict))

# The driver's canonical TransformerBridge-native hooks — what it actually serves:
fireable = sorted(bridge._driver.supported_hook_points)
print("fireable (native):", len(fireable))
print("layer 0:", [h for h in fireable if h.startswith("blocks.0.")])
print("non-fireable (use boot_transformers):", len(bridge._driver.non_fireable_hook_points))

# `list_hooks()` shows only hooks with a handler attached — empty until you cache/hook:
print("attached:", bridge.list_hooks())

hook_dict entries: 120
fireable (native): 60
layer 0: ['blocks.0.attn.hook_out', 'blocks.0.hook_in', 'blocks.0.hook_out', 'blocks.0.ln2.hook_in', 'blocks.0.mlp.hook_out']
non-fireable (use boot_transformers): 75
attached: {}


## Step 3 — `run_with_cache` over the Inspect boundary

The residual stream comes back through Inspect's `ModelOutput.metadata`, gets
converted to torch at the bridge boundary, and surfaces as a normal
`ActivationCache`.

In [4]:
tokens = bridge.to_tokens(PROMPT)
logits, cache = bridge.run_with_cache(tokens)
print("logits:", tuple(logits.shape))
hk = "blocks.0.hook_out"  # resid_post, TransformerBridge name
print(f"{hk}: shape={tuple(cache[hk].shape)} dtype={cache[hk].dtype}")
nxt = int(logits[0, -1].argmax())
print("next-token:", nxt, repr(bridge.tokenizer.decode([nxt])))

logits: (1, 5, 50257)
blocks.0.hook_out: shape=(1, 5, 768) dtype=torch.float32
next-token: 318 ' is'


## Step 4 — Parity vs `boot_transformers`

Our provider runs the same HF model, so the residual stream matches the local HF
bridge **exactly** (within fp32 noise). `RemoteBridge.to_tokens` is BOS-aware
(mirrors `boot_transformers`), so `bridge.run_with_cache("text")` would match too;
here we feed identical token ids to both so the only variable is the backend path.

In [5]:
hf = TransformerBridge.boot_transformers(MODEL, device="cpu")
toks = hf.to_tokens(PROMPT)                     # shared, BOS-prepended

hf_logits, hf_cache = hf.run_with_cache(toks)
_, i_cache = bridge.run_with_cache(toks)
i_logits = bridge.forward(toks)

print("argmax match:", int(hf_logits[0, -1].argmax()) == int(i_logits[0, -1].argmax()))
for hk in ["blocks.0.hook_out", "blocks.6.hook_out", "blocks.11.hook_out"]:  # resid_post
    a, b = hf_cache[hk].float(), i_cache[hk].float()
    ok = torch.allclose(a, b, atol=1e-3, rtol=1e-3)
    print(f"{hk}: allclose={ok}  maxdiff={(a - b).abs().max().item():.2e}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

argmax match: True
blocks.0.hook_out: allclose=True  maxdiff=3.81e-05
blocks.6.hook_out: allclose=True  maxdiff=2.44e-04
blocks.11.hook_out: allclose=True  maxdiff=1.53e-04


## Step 5 — Interventions

The full affine vocabulary is supported: `suppress` / `scale` / `add` / `set`.
Here we zero the residual stream after block 0 — it shows up zeroed in the cache
and shifts the prediction, and a clean run reverts (interventions aren't sticky).

In [6]:
clean = int(bridge.forward(toks)[0, -1].argmax())

hk = "blocks.0.hook_out"  # resid_post
supp_logits, supp_cache = bridge.run_with_cache(
    toks, intervene={hk: {"op": "suppress"}}
)
suppressed = int(supp_logits[0, -1].argmax())
print(f"{hk} |max| after suppress:", supp_cache[hk].abs().max().item())
print(f"argmax: clean={clean}  suppressed={suppressed}  changed={clean != suppressed}")

revert = int(bridge.forward(toks)[0, -1].argmax())
print(f"revert={revert}  matches clean={revert == clean}")

bridge.close()

blocks.0.hook_out |max| after suppress: 0.0
argmax: clean=318  suppressed=11  changed=True
revert=318  matches clean=True


## Inspect-native workflows

`tl_bridge/<model>` is also a normal `inspect_ai` model, so it plugs into the eval
pipeline — not just the bridge capture path above. The next cells show: **(6)** eval-native
generation with logprobs + usage, **(7)** capturing activations alongside a scored eval
(full tensors to a side-artifact + a reduction in `samples_df`), **(8)** per-turn capture
across a rollout, and **(9)** tool-aware generation.

In [7]:
# (6) Eval-native generation: tl_bridge is a normal Inspect model — real multi-token
# generation with token logprobs + usage, read straight from the eval log.
import tempfile

from inspect_ai import Task
from inspect_ai import eval as inspect_eval
from inspect_ai.dataset import Sample
from inspect_ai.solver import generate

run = tempfile.mkdtemp()
gen_log = inspect_eval(
    Task(dataset=[Sample(input="The capital of France is")], solver=generate()),
    model="tl_bridge/gpt2",
    max_tokens=6,
    logprobs=True,
    top_logprobs=3,
    log_dir=f"{run}/gen",
    display="none",
)[0]
out = gen_log.samples[0].output
print("completion:", repr(out.completion))
print("usage:", out.usage.input_tokens, "->", out.usage.output_tokens, "tokens")
top3 = out.choices[0].logprobs.content[0].top_logprobs
print("1st-token top-3:", [(t.token, round(t.logprob, 2)) for t in top3])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Output()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

completion: ' the capital of the French Republic'
usage: 5 -> 6 tokens
1st-token top-3: [(' the', -2.47), (' now', -3.04), (' a', -3.08)]


In [8]:
# (7) Capture activations alongside a scored eval: full tensors to a per-sample .npz,
# plus a reduction into the store surfaced by samples_df (to correlate with scores).
import pathlib

from inspect_ai.analysis import SampleSummary, samples_df
from inspect_ai.scorer import includes

from transformer_lens.model_bridge.sources.inspect import activations_column, capture_activations

cap_task = Task(
    dataset=[
        Sample(input="The capital of France is", target="Paris"),
        Sample(input="The opposite of hot is", target="cold"),
    ],
    solver=[capture_activations(["blocks.6.hook_out"], output_dir=f"{run}/acts"), generate()],
    scorer=includes(),
)
cap_log = inspect_eval(
    cap_task, model="tl_bridge/gpt2", max_tokens=4, log_dir=f"{run}/cap", display="none"
)[0]
print("status:", cap_log.status, "| npz:", [p.name for p in pathlib.Path(f"{run}/acts").glob("*.npz")])
df = samples_df(f"{run}/cap", columns=[*SampleSummary, activations_column()])
print(df[["id", "tl_activations"]].to_string(index=False))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Output()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


status: success | npz: ['2.npz', '1.npz']
id                                                      tl_activations
 1 {"blocks.6.hook_out": {"l2": 3052.146240234375, "shape": [5, 768]}}
 2   {"blocks.6.hook_out": {"l2": 3052.0810546875, "shape": [5, 768]}}


In [9]:
# (8) Per-turn capture: boot with capture=[...]; each model turn stashes its activations
# in the transcript, collected by turn_activations — the rollout's per-turn activations.
from transformer_lens.model_bridge.sources.inspect import turn_activations

turn_log = inspect_eval(
    Task(dataset=[Sample(input="The capital of France is")], solver=generate()),
    model="tl_bridge/gpt2",
    model_args={"capture": ["blocks.6.hook_out"]},
    max_tokens=4,
    log_dir=f"{run}/turns",
    display="none",
)[0]
turns = turn_activations(turn_log.samples[0])
print("model turns captured:", len(turns))
print("turn 0:", {k: tuple(v.shape) for k, v in turns[0].items()})

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Output()

model turns captured: 1
turn 0: {'blocks.6.hook_out': (1, 5, 768)}


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
# (9) Tool-aware generation: for a tool-capable instruct model the provider renders tool
# schemas into the chat template and parses tool calls from the output (so Inspect agent
# loops run). gpt2 has no tool template, so here we just show the tool-call parser:
from transformer_lens.model_bridge.sources.inspect.transformers_provider import _parse_tool_calls

calls = _parse_tool_calls('<tool_call>{"name": "add", "arguments": {"a": 2, "b": 2}}</tool_call>')
print("parsed tool calls:", [(c.function, c.arguments) for c in calls])

## Summary

- `boot_inspect("gpt2")` returns a bridge with the standard TL hook contract.
- Residual / attention / MLP activations cross the Inspect boundary and match
  `boot_transformers` to fp tolerance (exact next-token argmax) — conversion to torch
  happens at the bridge.
- Full-sequence logits, so `return_type="loss"` matches `boot_transformers` too.
- Full affine interventions (suppress/scale/add/set) apply and revert.
- `tl_bridge/<model>` is also a normal Inspect model: eval-native generation with
  logprobs + usage, `capture_activations` alongside a scored eval (→ `.npz` + `samples_df`),
  per-turn capture across rollouts (`turn_activations`), and tool-aware generation.

**Out of v1 scope:** head-split hooks (`hook_q/k/v/z`, `hook_pattern`), `embed`/`ln_final`
(convention), batch > 1, and live agent loops with tool *execution* (need a tool-capable
model). For those, use `boot_transformers()`.